# Test I: Multi-Class Classification — Classical CNN

**Task:** Classify strong gravitational lensing images into 3 classes:
- `no` — no substructure
- `sphere` — subhalo substructure  
- `vort` — vortex substructure

**Approach:** ResNet-18 backbone adapted for single-channel 150×150 input.

**Evaluation:** ROC curve + AUC score (one-vs-rest)

In [ ]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc, roc_auc_score
from sklearn.preprocessing import label_binarize
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

## 1. Dataset

In [ ]:
CLASS_NAMES = ['no', 'sphere', 'vort']
CLASS_TO_IDX = {name: i for i, name in enumerate(CLASS_NAMES)}

class LensingDataset(Dataset):
    def __init__(self, root_dir, augment=False):
        self.samples = []
        self.labels = []
        self.augment = augment
        for class_name in CLASS_NAMES:
            class_dir = os.path.join(root_dir, class_name)
            for fname in sorted(os.listdir(class_dir)):
                if fname.endswith('.npy'):
                    self.samples.append(os.path.join(class_dir, fname))
                    self.labels.append(CLASS_TO_IDX[class_name])
        self.labels = np.array(self.labels)
        print(f'Loaded {len(self.samples)} samples from {root_dir}')

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img = np.load(self.samples[idx]).astype(np.float32)  # (1, 150, 150)
        label = self.labels[idx]
        if self.augment:
            # Random horizontal/vertical flip
            if np.random.rand() > 0.5:
                img = img[:, :, ::-1].copy()
            if np.random.rand() > 0.5:
                img = img[:, ::-1, :].copy()
            # Random 90-degree rotation
            k = np.random.randint(0, 4)
            img = np.rot90(img, k, axes=(1, 2)).copy()
        return torch.from_numpy(img), label

In [ ]:
train_dataset = LensingDataset('dataset/dataset/train', augment=True)
val_dataset = LensingDataset('dataset/dataset/val', augment=False)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False, num_workers=4, pin_memory=True)

In [ ]:
# Visualize samples
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for i, class_name in enumerate(CLASS_NAMES):
    idx = np.where(train_dataset.labels == i)[0][0]
    img, _ = train_dataset[idx]
    axes[i].imshow(img[0], cmap='hot')
    axes[i].set_title(class_name)
    axes[i].axis('off')
plt.tight_layout()
plt.show()

## 2. Model — ResNet-18

In [ ]:
def build_resnet18(num_classes=3):
    model = models.resnet18(weights=None)
    # Adapt first conv for single-channel input
    model.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model

model = build_resnet18().to(device)
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

## 3. Training

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=30)

NUM_EPOCHS = 30
best_val_acc = 0.0
history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

for epoch in range(NUM_EPOCHS):
    # Train
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * imgs.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += imgs.size(0)
    train_loss = running_loss / total
    train_acc = correct / total

    # Validate
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            running_loss += loss.item() * imgs.size(0)
            correct += (outputs.argmax(1) == labels).sum().item()
            total += imgs.size(0)
    val_loss = running_loss / total
    val_acc = correct / total

    scheduler.step()
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), 'best_cnn.pt')

    print(f'Epoch {epoch+1:02d}/{NUM_EPOCHS} | '
          f'Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | '
          f'Val Loss: {val_loss:.4f} Acc: {val_acc:.4f}')

print(f'\nBest Val Accuracy: {best_val_acc:.4f}')

In [ ]:
# Training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(history['train_loss'], label='Train')
ax1.plot(history['val_loss'], label='Val')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss'); ax1.legend(); ax1.set_title('Loss')
ax2.plot(history['train_acc'], label='Train')
ax2.plot(history['val_acc'], label='Val')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy'); ax2.legend(); ax2.set_title('Accuracy')
plt.tight_layout()
plt.show()

## 4. Evaluation — ROC Curve & AUC

In [ ]:
# Load best model
model.load_state_dict(torch.load('best_cnn.pt'))
model.eval()

all_probs = []
all_labels = []
with torch.no_grad():
    for imgs, labels in val_loader:
        imgs = imgs.to(device)
        outputs = model(imgs)
        probs = torch.softmax(outputs, dim=1).cpu().numpy()
        all_probs.append(probs)
        all_labels.append(labels.numpy())

all_probs = np.concatenate(all_probs)
all_labels = np.concatenate(all_labels)
all_labels_bin = label_binarize(all_labels, classes=[0, 1, 2])

In [ ]:
# Plot ROC curves (one-vs-rest)
fig, ax = plt.subplots(figsize=(8, 6))
colors = ['#e41a1c', '#377eb8', '#4daf4a']

for i, class_name in enumerate(CLASS_NAMES):
    fpr, tpr, _ = roc_curve(all_labels_bin[:, i], all_probs[:, i])
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, color=colors[i], lw=2, label=f'{class_name} (AUC = {roc_auc:.4f})')

# Macro average
macro_auc = roc_auc_score(all_labels_bin, all_probs, multi_class='ovr', average='macro')
ax.plot([0, 1], [0, 1], 'k--', lw=1)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title(f'ROC Curves — Classical CNN (Macro AUC = {macro_auc:.4f})')
ax.legend(loc='lower right')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f'Macro AUC: {macro_auc:.4f}')